1. WESAD pkl에서 BVP 읽기
2. raw BVP에 noise level별 Gaussian noise 추가
3. 기존 TPV 방식 그대로 TPV22 추출
4. 기존 HRV 방식 그대로 1분 RMSSD 추출
5. noise level별로 결과 CSV 저장
6. TPV22 / RMSSD 박스플롯 저장
7. noise level별 ANOVA, Kruskal, effect size, consistency ratio 계산
8. 최종 summary CSV 저장

In [ ]:
!pip install ripser

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.1/842.1 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 4.2 MB/s eta 0:00:00
  Created wheel for hopcroftkarp: filename=hopcroftkarp-1.2.5-py2.py3-none-any.whl size=18104 sha256=7f0195c7d149444247196804d937421d12189eec53287897240d9484f5295cf7
  Stored in directory: /root/.cache/pip/wheels/2a/fd/fe/f4b8fd82894e1d9e04040ef41dc5ae6eb7a8e9b0ef5a9402fe
Successfully built hopcroftkarp


In [ ]:
import os
import glob
import pickle
import warnings
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ripser import ripser
from scipy.signal import butter, filtfilt, find_peaks, welch
from scipy.stats import iqr, skew, kurtosis, entropy, f_oneway, kruskal
from scipy.spatial.distance import pdist
from scipy.interpolate import interp1d

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIG
# =============================================================================
PKL_DIR = r"/content/drive/MyDrive/Colab Notebooks/BP/data/WESAD/pkl_data"
OUTPUT_DIR = r"/content/drive/MyDrive/Colab Notebooks/BP/noise_robustness_TPV22_RMSSD"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 기존 orig csv 경로
ORIG_TPV_CSV = r"/content/drive/MyDrive/Colab Notebooks/BP/TPV-csv/noQC.csv"
ORIG_HRV_CSV = r"/content/drive/MyDrive/Colab Notebooks/BP/HRV-csv/HRV_1min.csv"

FS_LABEL = 700
FS_BVP = 64
FS_ACC = 32

WINDOW_SEC = 60
STEP_SEC = 60

LABEL_BASELINE = 1
LABEL_STRESS = 2
LABEL_MEDITATION = 4

STATUS_MAP = {
    LABEL_BASELINE: 0,
    LABEL_STRESS: 1,
    LABEL_MEDITATION: 2
}
STATUS_NAME_MAP = {
    LABEL_BASELINE: "baseline",
    LABEL_STRESS: "stress",
    LABEL_MEDITATION: "meditation"
}

# orig는 csv에서 불러오므로 noise 추출 대상에서는 제외
NOISE_LEVELS = {
    "alpha_0.01": 0.01,
    "alpha_0.03": 0.03,
    "alpha_0.05": 0.05,
}

RANDOM_SEED = 42

LOWCUT = 0.5
HIGHCUT = 8.0
FILTER_ORDER = 4

MAJORITY_RATIO_MIN = 0.95

# peak detection / HRV
MIN_HR_BPM = 40
MAX_HR_BPM = 180
MIN_PEAK_DISTANCE_SEC = 60.0 / MAX_HR_BPM
IBI_MIN_SEC = 0.30
IBI_MAX_SEC = 1.50

# TPV
TPV_FEATURE_NAMES = [
    "birth_mean", "birth_std", "death_mean", "death_std",
    "lifetime_mean", "lifetime_std", "lifetime_max", "lifetime_min",
    "lifetime_median", "lifetime_iqr", "lifetime_skew", "lifetime_kurtosis",
    "birth_max", "birth_min", "birth_median", "birth_skew", "birth_kurtosis",
    "death_max", "death_min", "death_median", "death_skew", "death_kurtosis",
    "num_lifetimes", "top1_lifetime", "top2_lifetime", "lifetime_max_div_min",
    "ph_entropy", "betti_entropy", "avg_persistence_distance", "gini_index",
    "lifetime_variance", "persistent_image_energy",
]
N_FEATURES = len(TPV_FEATURE_NAMES)

# HRV frequency domain
FS_IBI_INTERP = 4.0
LF_LOW = 0.04
LF_HIGH = 0.15
HF_LOW = 0.15
HF_HIGH = 0.40
IBI_DIFF_RATIO_MAX = 0.30


# =============================================================================
# BASIC UTILS
# =============================================================================
def fill_nan_with_median(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32).reshape(-1).copy()
    if np.isnan(x).any():
        med = np.nanmedian(x)
        if np.isnan(med):
            med = 0.0
        x = np.nan_to_num(x, nan=med, posinf=med, neginf=med)
    return x


def bandpass_filter(sig, fs=64, low=0.5, high=8.0, order=4):
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    nyq = 0.5 * fs
    low_n = low / nyq
    high_n = high / nyq

    if not (0 < low_n < high_n < 1):
        raise ValueError(f"Invalid bandpass range: low={low}, high={high}, fs={fs}")

    b, a = butter(order, [low_n, high_n], btype="band")
    return filtfilt(b, a, sig).astype(np.float32)


def zscore_1d(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    mu = float(np.mean(x))
    sd = float(np.std(x))
    if sd < 1e-8:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - mu) / (sd + 1e-8)).astype(np.float32)


def safe_skew(x: np.ndarray) -> float:
    if len(x) > 2 and np.std(x) > 1e-12:
        v = skew(x)
        return float(v) if np.isfinite(v) else 0.0
    return 0.0


def safe_kurtosis(x: np.ndarray) -> float:
    if len(x) > 3 and np.std(x) > 1e-12:
        v = kurtosis(x)
        return float(v) if np.isfinite(v) else 0.0
    return 0.0


def safe_div(a, b):
    return float(a / b) if abs(b) > 1e-12 else np.nan


def add_gaussian_noise(sig: np.ndarray, alpha: float, rng: np.random.Generator) -> np.ndarray:
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    if alpha <= 0:
        return sig.copy()

    sig_std = float(np.std(sig))
    if sig_std < 1e-8:
        return sig.copy()

    noise = rng.normal(loc=0.0, scale=alpha * sig_std, size=len(sig)).astype(np.float32)
    return (sig + noise).astype(np.float32)


# =============================================================================
# TARGET BLOCKS
# =============================================================================
def find_target_blocks(labels: np.ndarray):
    labels = np.asarray(labels).reshape(-1)
    if len(labels) == 0:
        return {}

    segments = []
    start = 0
    cur = labels[0]
    for i in range(1, len(labels)):
        if labels[i] != cur:
            segments.append((int(cur), start, i))
            start = i
            cur = labels[i]
    segments.append((int(cur), start, len(labels)))

    baseline_seg = None
    stress_seg = None
    meditation_seg = None

    for lab, s, e in segments:
        if lab == LABEL_BASELINE:
            baseline_seg = (s, e)
            break

    for lab, s, e in segments:
        if lab == LABEL_STRESS:
            stress_seg = (s, e)
            break

    for lab, s, e in reversed(segments):
        if lab == LABEL_MEDITATION:
            meditation_seg = (s, e)
            break

    out = {}
    if baseline_seg is not None:
        out[LABEL_BASELINE] = baseline_seg
    if stress_seg is not None:
        out[LABEL_STRESS] = stress_seg
    if meditation_seg is not None:
        out[LABEL_MEDITATION] = meditation_seg
    return out


def window_fully_in_block(start_l: int, end_l: int, label_major: int, target_blocks: dict) -> bool:
    if label_major not in target_blocks:
        return False
    blk_s, blk_e = target_blocks[label_major]
    return (start_l >= blk_s) and (end_l <= blk_e)


# =============================================================================
# TPV EXTRACTION
# =============================================================================
def extract_tpv_features(sig_1d: np.ndarray) -> np.ndarray:
    sig = np.asarray(sig_1d, dtype=np.float32).reshape(-1)

    if len(sig) < 3:
        return np.zeros(N_FEATURES, dtype=np.float32)

    s = float(np.std(sig))
    if s < 1e-8:
        return np.zeros(N_FEATURES, dtype=np.float32)

    sig = (sig - float(np.mean(sig))) / (s + 1e-8)

    X = np.stack([sig[:-1], sig[1:]], axis=1)
    dgms = ripser(X, maxdim=1)["dgms"]
    H1 = dgms[1]

    if H1.size == 0:
        return np.zeros(N_FEATURES, dtype=np.float32)

    births = H1[:, 0]
    deaths = H1[:, 1]
    lifetimes = deaths - births

    mask = np.isfinite(births) & np.isfinite(deaths) & np.isfinite(lifetimes)
    births = births[mask]
    deaths = deaths[mask]
    lifetimes = lifetimes[mask]

    if len(lifetimes) == 0:
        return np.zeros(N_FEATURES, dtype=np.float32)

    lifetimes_sorted = np.sort(lifetimes)
    lifetime_sum = float(np.sum(lifetimes))
    lifetime_ratio = lifetimes / (lifetime_sum + 1e-8)
    ph_entropy = float(-np.sum(lifetime_ratio * np.log(lifetime_ratio + 1e-10)))

    bmin = float(np.min(births))
    bmax = float(np.max(births))
    if bmax - bmin < 1e-8:
        betti_entropy = 0.0
    else:
        hist, _ = np.histogram(births, bins=10, range=(bmin, bmax), density=True)
        betti_entropy = float(entropy(hist + 1e-10))

    avg_persistence_distance = (
        float(np.mean(pdist(lifetimes[:, None]))) if len(lifetimes) > 1 else 0.0
    )

    if np.sum(lifetimes_sorted) > 0 and len(lifetimes_sorted) > 1:
        n = len(lifetimes_sorted)
        gini = (
            2 * np.sum(np.arange(1, n + 1) * lifetimes_sorted)
        ) / (n * np.sum(lifetimes_sorted)) - (n + 1) / n
        gini_index = float(gini)
    else:
        gini_index = 0.0

    lifetime_variance = float(np.var(lifetimes))
    persistent_image_energy = float(np.sum(lifetimes ** 2))

    feats = [
        float(np.mean(births)),
        float(np.std(births)),
        float(np.mean(deaths)),
        float(np.std(deaths)),
        float(np.mean(lifetimes)),
        float(np.std(lifetimes)),
        float(np.max(lifetimes)),
        float(np.min(lifetimes)),
        float(np.median(lifetimes)),
        float(iqr(lifetimes)),
        safe_skew(lifetimes),
        safe_kurtosis(lifetimes),
        float(np.max(births)),
        float(np.min(births)),
        float(np.median(births)),
        safe_skew(births),
        safe_kurtosis(births),
        float(np.max(deaths)),
        float(np.min(deaths)),
        float(np.median(deaths)),
        safe_skew(deaths),
        safe_kurtosis(deaths),
        float(len(lifetimes)),
        float(lifetimes_sorted[-1]),
        float(lifetimes_sorted[-2]) if len(lifetimes_sorted) > 1 else 0.0,
        float(np.max(lifetimes) / (np.min(lifetimes) + 1e-8)),
        ph_entropy,
        betti_entropy,
        avg_persistence_distance,
        gini_index,
        lifetime_variance,
        persistent_image_energy,
    ]
    return np.asarray(feats, dtype=np.float32)


def preprocess_bvp_for_tpv(seg_bvp_raw: np.ndarray, fs_bvp: int) -> np.ndarray:
    seg_bvp = fill_nan_with_median(seg_bvp_raw)
    seg_bvp_bp = bandpass_filter(seg_bvp, fs=fs_bvp, low=LOWCUT, high=HIGHCUT, order=FILTER_ORDER)
    seg_bvp_z = zscore_1d(seg_bvp_bp)
    return seg_bvp_z


# =============================================================================
# HRV EXTRACTION
# =============================================================================
def detect_ppg_peaks(sig: np.ndarray, fs: int):
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    distance = max(1, int(round(MIN_PEAK_DISTANCE_SEC * fs)))
    prom = max(0.10, 0.20 * float(np.std(sig)))
    peaks, props = find_peaks(sig, distance=distance, prominence=prom)
    return peaks, props


def compute_basic_peak_info(peaks: np.ndarray, fs: int):
    peaks = np.asarray(peaks, dtype=np.int64)
    if len(peaks) < 2:
        return {
            "n_peaks": int(len(peaks)),
            "ibi_mean_sec_raw": np.nan,
            "ibi_pass_ratio": 0.0,
            "valid_ibi_count": 0,
        }

    ibi_raw = np.diff(peaks) / float(fs)
    plausible = (ibi_raw >= IBI_MIN_SEC) & (ibi_raw <= IBI_MAX_SEC)
    ibi_pass_ratio = float(np.mean(plausible)) if len(ibi_raw) > 0 else 0.0

    return {
        "n_peaks": int(len(peaks)),
        "ibi_mean_sec_raw": float(np.mean(ibi_raw)) if len(ibi_raw) > 0 else np.nan,
        "ibi_pass_ratio": ibi_pass_ratio,
        "valid_ibi_count": int(np.sum(plausible)),
    }


def build_filtered_ibi_series(peaks: np.ndarray, fs: int):
    peaks = np.asarray(peaks, dtype=np.int64)

    if len(peaks) < 3:
        return None

    peak_times = peaks.astype(np.float64) / float(fs)
    ibi = np.diff(peak_times)
    ibi_times = peak_times[1:]

    if len(ibi) < 2:
        return None

    plausible = (ibi >= IBI_MIN_SEC) & (ibi <= IBI_MAX_SEC)
    ibi = ibi[plausible]
    ibi_times = ibi_times[plausible]

    if len(ibi) < 2:
        return None

    med_ibi = float(np.median(ibi))
    stable = np.abs(ibi - med_ibi) <= (IBI_DIFF_RATIO_MAX * (med_ibi + 1e-8))

    ibi = ibi[stable]
    ibi_times = ibi_times[stable]

    if len(ibi) < 2:
        return None

    return {
        "peak_times": peak_times,
        "ibi_times": ibi_times,
        "ibi": ibi,
        "ibi_median_sec": med_ibi,
    }


def interpolate_ibi(ibi_times: np.ndarray, ibi: np.ndarray, fs_interp: float = FS_IBI_INTERP):
    ibi_times = np.asarray(ibi_times, dtype=np.float64).reshape(-1)
    ibi = np.asarray(ibi, dtype=np.float64).reshape(-1)

    if len(ibi_times) < 2 or len(ibi) < 2:
        return None

    if ibi_times[-1] <= ibi_times[0]:
        return None

    t_uniform = np.arange(ibi_times[0], ibi_times[-1], 1.0 / fs_interp)
    if len(t_uniform) < 4:
        return None

    try:
        f_interp = interp1d(
            ibi_times,
            ibi,
            kind="linear",
            bounds_error=False,
            fill_value="extrapolate",
            assume_sorted=True
        )
        ibi_uniform = f_interp(t_uniform)
    except Exception:
        return None

    ibi_uniform = np.asarray(ibi_uniform, dtype=np.float64)
    if np.any(~np.isfinite(ibi_uniform)):
        return None

    return {
        "t_uniform": t_uniform,
        "ibi_uniform": ibi_uniform,
    }


def compute_freq_domain_hrv(ibi_uniform: np.ndarray, fs_interp: float = FS_IBI_INTERP):
    ibi_uniform = np.asarray(ibi_uniform, dtype=np.float64).reshape(-1)

    if len(ibi_uniform) < 8:
        return {
            "LF": np.nan,
            "HF": np.nan,
            "LF_HF": np.nan,
        }

    nperseg = min(256, len(ibi_uniform))
    f, pxx = welch(ibi_uniform, fs=fs_interp, nperseg=nperseg)

    lf_mask = (f >= LF_LOW) & (f < LF_HIGH)
    hf_mask = (f >= HF_LOW) & (f < HF_HIGH)

    lf = np.trapz(pxx[lf_mask], f[lf_mask]) if np.any(lf_mask) else np.nan
    hf = np.trapz(pxx[hf_mask], f[hf_mask]) if np.any(hf_mask) else np.nan
    lf_hf = safe_div(lf, hf) if np.isfinite(lf) and np.isfinite(hf) else np.nan

    return {
        "LF": float(lf) if np.isfinite(lf) else np.nan,
        "HF": float(hf) if np.isfinite(hf) else np.nan,
        "LF_HF": float(lf_hf) if np.isfinite(lf_hf) else np.nan,
    }


def compute_time_domain_hrv(ibi: np.ndarray):
    ibi = np.asarray(ibi, dtype=np.float64).reshape(-1)

    if len(ibi) < 2:
        return {
            "RMSSD": np.nan,
            "SDNN": np.nan,
            "ibi_mean_sec": np.nan,
            "ibi_std_sec": np.nan,
        }

    diff_ibi = np.diff(ibi)

    rmssd = np.sqrt(np.mean(diff_ibi ** 2)) if len(diff_ibi) > 0 else np.nan
    sdnn = np.std(ibi, ddof=0)

    return {
        "RMSSD": float(rmssd) if np.isfinite(rmssd) else np.nan,
        "SDNN": float(sdnn) if np.isfinite(sdnn) else np.nan,
        "ibi_mean_sec": float(np.mean(ibi)) if np.isfinite(np.mean(ibi)) else np.nan,
        "ibi_std_sec": float(np.std(ibi, ddof=0)) if np.isfinite(np.std(ibi, ddof=0)) else np.nan,
    }


def extract_hrv_features_from_bvp_segment(seg_bvp_raw: np.ndarray, fs_bvp: int):
    seg_bvp = fill_nan_with_median(seg_bvp_raw)
    seg_bvp_bp = bandpass_filter(seg_bvp, fs=fs_bvp, low=LOWCUT, high=HIGHCUT, order=FILTER_ORDER)
    seg_bvp_z = zscore_1d(seg_bvp_bp)

    peaks, props = detect_ppg_peaks(seg_bvp_z, fs=fs_bvp)
    peak_info = compute_basic_peak_info(peaks, fs_bvp)

    ibi_pack = build_filtered_ibi_series(peaks, fs_bvp)

    row = {
        "n_peaks": peak_info["n_peaks"],
        "ibi_pass_ratio": peak_info["ibi_pass_ratio"],
        "valid_ibi_count": 0,
        "ibi_mean_sec_raw": peak_info["ibi_mean_sec_raw"],
        "ibi_mean_sec": np.nan,
        "ibi_std_sec": np.nan,
        "RMSSD": np.nan,
        "SDNN": np.nan,
        "LF": np.nan,
        "HF": np.nan,
        "LF_HF": np.nan,
    }

    if ibi_pack is None:
        return row

    ibi = ibi_pack["ibi"]
    ibi_times = ibi_pack["ibi_times"]

    row["valid_ibi_count"] = int(len(ibi))

    td = compute_time_domain_hrv(ibi)
    row.update(td)

    interp_pack = interpolate_ibi(ibi_times, ibi, fs_interp=FS_IBI_INTERP)
    if interp_pack is None:
        return row

    fd = compute_freq_domain_hrv(interp_pack["ibi_uniform"], fs_interp=FS_IBI_INTERP)
    row.update(fd)

    return row


# =============================================================================
# LOAD ORIGINAL CSVs
# =============================================================================
def load_original_feature_csvs(tpv_csv_path: str, hrv_csv_path: str) -> pd.DataFrame:
    df_tpv = pd.read_csv(tpv_csv_path)
    df_hrv = pd.read_csv(hrv_csv_path)

    tpv_cols = [
        "subject", "window", "window_index", "status", "status_name",
        "label_major", "t_start_sec", "t_end_sec", "major_ratio", "TPV22"
    ]
    hrv_cols = [
        "subject", "window", "window_index", "status", "status_name",
        "label_major", "t_start_sec", "t_end_sec", "major_ratio", "RMSSD"
    ]

    df_tpv = df_tpv[tpv_cols].copy()
    df_hrv = df_hrv[hrv_cols].copy()

    df_orig = pd.merge(
        df_tpv,
        df_hrv,
        on=[
            "subject", "window", "window_index", "status", "status_name",
            "label_major", "t_start_sec", "t_end_sec", "major_ratio"
        ],
        how="inner"
    )

    df_orig["noise"] = "orig"
    df_orig["alpha"] = 0.0
    return df_orig


# =============================================================================
# BUILD NOISY FEATURES ONLY
# =============================================================================
def build_subject_feature_table_with_noise(pkl_path: str, subject_name: str, noise_name: str, alpha: float, rng):
    with open(pkl_path, "rb") as f:
        s = pickle.load(f, encoding="latin1")

    bvp = np.asarray(s["signal"]["wrist"]["BVP"]).reshape(-1).astype(np.float32)
    labels = np.asarray(s["label"]).reshape(-1).astype(np.int64)

    dur_wrist = len(bvp) / FS_BVP
    labels = labels[: int(dur_wrist * FS_LABEL)]

    bvp = fill_nan_with_median(bvp)
    bvp_noisy = add_gaussian_noise(bvp, alpha=alpha, rng=rng)

    target_blocks = find_target_blocks(labels)
    if len(target_blocks) == 0:
        return pd.DataFrame()

    Wl = int(WINDOW_SEC * FS_LABEL)
    Sl = int(STEP_SEC * FS_LABEL)
    Wb = int(WINDOW_SEC * FS_BVP)

    rows = []
    window_counter = 0

    for start_l in range(0, len(labels) - Wl + 1, Sl):
        end_l = start_l + Wl
        win_lab = labels[start_l:end_l]

        binc = np.bincount(win_lab)
        maj = int(np.argmax(binc))
        maj_ratio = float((win_lab == maj).mean())

        if maj not in [LABEL_BASELINE, LABEL_STRESS, LABEL_MEDITATION]:
            continue
        if maj_ratio < MAJORITY_RATIO_MIN:
            continue
        if not window_fully_in_block(start_l, end_l, maj, target_blocks):
            continue

        t0 = start_l / FS_LABEL
        t1 = t0 + WINDOW_SEC

        start_b = int(round(t0 * FS_BVP))
        end_b = start_b + Wb

        if end_b > len(bvp_noisy):
            continue

        seg_bvp_raw = bvp_noisy[start_b:end_b]
        if len(seg_bvp_raw) != Wb:
            continue

        # TPV no-QC pipeline
        seg_bvp_for_tpv = preprocess_bvp_for_tpv(seg_bvp_raw, FS_BVP)
        tpv_full = extract_tpv_features(seg_bvp_for_tpv)
        tpv22 = float(tpv_full[22])

        # HRV 1-min RMSSD pipeline
        hrv_info = extract_hrv_features_from_bvp_segment(seg_bvp_raw, FS_BVP)
        rmssd = hrv_info["RMSSD"]

        window_counter += 1

        row = {
            "subject": subject_name,
            "window": f"window{window_counter}",
            "window_index": window_counter,
            "noise": noise_name,
            "alpha": alpha,
            "status": STATUS_MAP[maj],
            "status_name": STATUS_NAME_MAP[maj],
            "label_major": maj,
            "t_start_sec": float(t0),
            "t_end_sec": float(t1),
            "major_ratio": maj_ratio,
            "TPV22": tpv22,
            "RMSSD": rmssd,
        }
        rows.append(row)

    return pd.DataFrame(rows)


def build_all_subjects_all_noise(pkl_dir: str):
    pkl_list = sorted(glob.glob(os.path.join(pkl_dir, "S*.pkl")))
    all_dfs = []

    print(f"[INFO] Found {len(pkl_list)} pkl files")

    for noise_name, alpha in NOISE_LEVELS.items():
        print(f"\n{'='*80}")
        print(f"[INFO] Noise level: {noise_name} (alpha={alpha})")
        print(f"{'='*80}")

        for pkl_path in pkl_list:
            subject_name = os.path.splitext(os.path.basename(pkl_path))[0]

            seed_str = f"{subject_name}_{noise_name}_{RANDOM_SEED}"
            seed_val = int(hashlib.md5(seed_str.encode()).hexdigest()[:8], 16)
            rng = np.random.default_rng(seed_val)

            print(f"[INFO] Processing {subject_name} ...")
            try:
                df_sub = build_subject_feature_table_with_noise(
                    pkl_path=pkl_path,
                    subject_name=subject_name,
                    noise_name=noise_name,
                    alpha=alpha,
                    rng=rng
                )
                print(f"[INFO] {subject_name}: extracted {len(df_sub)} windows")
                if len(df_sub) > 0:
                    all_dfs.append(df_sub)
            except Exception as e:
                print(f"[WARN] Failed on {subject_name} @ {noise_name}: {e}")

    if len(all_dfs) == 0:
        return pd.DataFrame()

    return pd.concat(all_dfs, axis=0, ignore_index=True)


# =============================================================================
# STATS
# =============================================================================
def eta_squared_anova(groups):
    all_vals = np.concatenate(groups)
    grand_mean = np.mean(all_vals)

    ss_between = 0.0
    ss_total = np.sum((all_vals - grand_mean) ** 2)

    for g in groups:
        if len(g) == 0:
            continue
        ss_between += len(g) * (np.mean(g) - grand_mean) ** 2

    if ss_total == 0:
        return np.nan
    return ss_between / ss_total


def consistency_ratio(df, value_col, label_col="status_name"):
    grouped = df.groupby(label_col)[value_col]
    means = grouped.mean()
    stds = grouped.std()

    if len(means) < 2:
        return np.nan

    denom = stds.mean()
    if pd.isna(denom) or denom == 0:
        return np.nan

    return means.std() / denom


def summarize_by_noise_and_feature(df, value_col):
    results = []
    order_labels = ["baseline", "stress", "meditation"]
    order_noise = ["orig"] + list(NOISE_LEVELS.keys())

    for noise_name in order_noise:
        sub = df[df["noise"] == noise_name].copy()
        sub = sub[np.isfinite(sub[value_col])].copy()

        groups = []
        label_used = []
        means = {}
        stds = {}
        ns = {}

        for lb in order_labels:
            vals = sub.loc[sub["status_name"] == lb, value_col].dropna().values
            if len(vals) > 0:
                groups.append(vals)
                label_used.append(lb)
                means[lb] = float(np.mean(vals))
                stds[lb] = float(np.std(vals, ddof=0))
                ns[lb] = int(len(vals))

        alpha_val = 0.0 if noise_name == "orig" else NOISE_LEVELS[noise_name]

        row = {
            "feature": value_col,
            "noise": noise_name,
            "alpha": alpha_val,
            "n_groups": len(groups),
            "labels_used": ",".join(label_used),
            "anova_F": np.nan,
            "anova_p": np.nan,
            "kruskal_H": np.nan,
            "kruskal_p": np.nan,
            "eta_sq": np.nan,
            "consistency_ratio": np.nan,
            "baseline_n": ns.get("baseline", 0),
            "stress_n": ns.get("stress", 0),
            "meditation_n": ns.get("meditation", 0),
            "baseline_mean": means.get("baseline", np.nan),
            "stress_mean": means.get("stress", np.nan),
            "meditation_mean": means.get("meditation", np.nan),
            "baseline_std": stds.get("baseline", np.nan),
            "stress_std": stds.get("stress", np.nan),
            "meditation_std": stds.get("meditation", np.nan),
        }

        if len(groups) >= 2:
            try:
                f_stat, p_val = f_oneway(*groups)
                row["anova_F"] = float(f_stat)
                row["anova_p"] = float(p_val)
            except Exception:
                pass

            try:
                h_stat, p_kw = kruskal(*groups)
                row["kruskal_H"] = float(h_stat)
                row["kruskal_p"] = float(p_kw)
            except Exception:
                pass

            try:
                row["eta_sq"] = float(eta_squared_anova(groups))
            except Exception:
                pass

            try:
                row["consistency_ratio"] = float(consistency_ratio(sub, value_col))
            except Exception:
                pass

        results.append(row)

    return pd.DataFrame(results)


# =============================================================================
# PLOTS
# =============================================================================
def save_boxplots(df):
    os.makedirs(os.path.join(OUTPUT_DIR, "plots"), exist_ok=True)

    order_noise = ["orig"] + list(NOISE_LEVELS.keys())
    order_status = ["baseline", "stress", "meditation"]

    for feature in ["TPV22", "RMSSD"]:
        sub = df[np.isfinite(df[feature])].copy()

        plt.figure(figsize=(12, 6))
        sns.boxplot(
            data=sub,
            x="noise",
            y=feature,
            hue="status_name",
            order=order_noise,
            hue_order=order_status
        )
        plt.title(f"{feature} distribution by condition across noise levels")
        plt.xlabel("Noise level")
        plt.ylabel(feature)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "plots", f"boxplot_{feature}_by_noise.png"), dpi=300)
        plt.close()

    fig, axes = plt.subplots(1, len(order_noise), figsize=(5.5 * len(order_noise), 5), sharey=False)
    for i, noise_name in enumerate(order_noise):
        ax = axes[i]
        sub = df[(df["noise"] == noise_name) & np.isfinite(df["TPV22"])].copy()
        sns.boxplot(
            data=sub,
            x="status_name",
            y="TPV22",
            order=order_status,
            ax=ax
        )
        ax.set_title(f"TPV22 - {noise_name}")
        ax.set_xlabel("")
        ax.set_ylabel("TPV22")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "plots", "boxplot_TPV22_panels.png"), dpi=300)
    plt.close()

    fig, axes = plt.subplots(1, len(order_noise), figsize=(5.5 * len(order_noise), 5), sharey=False)
    for i, noise_name in enumerate(order_noise):
        ax = axes[i]
        sub = df[(df["noise"] == noise_name) & np.isfinite(df["RMSSD"])].copy()
        sns.boxplot(
            data=sub,
            x="status_name",
            y="RMSSD",
            order=order_status,
            ax=ax
        )
        ax.set_title(f"RMSSD - {noise_name}")
        ax.set_xlabel("")
        ax.set_ylabel("RMSSD")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "plots", "boxplot_RMSSD_panels.png"), dpi=300)
    plt.close()


def save_robustness_lineplots(stats_df):
    os.makedirs(os.path.join(OUTPUT_DIR, "plots"), exist_ok=True)

    x_order = ["orig"] + list(NOISE_LEVELS.keys())

    for metric in ["anova_p", "eta_sq", "consistency_ratio"]:
        plt.figure(figsize=(8, 5))
        for feature in ["TPV22", "RMSSD"]:
            sub = stats_df[stats_df["feature"] == feature].copy()
            sub["noise"] = pd.Categorical(sub["noise"], categories=x_order, ordered=True)
            sub = sub.sort_values("noise")
            plt.plot(sub["noise"].astype(str), sub[metric], marker="o", label=feature)

        plt.title(f"Robustness comparison: {metric}")
        plt.xlabel("Noise level")
        plt.ylabel(metric)
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "plots", f"robustness_{metric}.png"), dpi=300)
        plt.close()


# =============================================================================
# MAIN
# =============================================================================
if __name__ == "__main__":
    # orig는 기존 CSV에서 불러옴
    df_orig = load_original_feature_csvs(ORIG_TPV_CSV, ORIG_HRV_CSV)
    print(f"[INFO] Loaded original CSV merged shape: {df_orig.shape}")

    # noisy만 새로 추출
    df_noise = build_all_subjects_all_noise(PKL_DIR)
    print(f"[INFO] Loaded noisy extracted shape: {df_noise.shape}")

    if len(df_orig) == 0 and len(df_noise) == 0:
        raise SystemExit("[WARN] No data available.")

    df_all = pd.concat([df_orig, df_noise], ignore_index=True)

    print("\n[INFO] Final merged shape:", df_all.shape)

    raw_csv = os.path.join(OUTPUT_DIR, "TPV22_RMSSD_all_windows_with_orig_and_noise.csv")
    df_all.to_csv(raw_csv, index=False)
    print(f"[INFO] Saved merged feature CSV: {raw_csv}")

    # noisy 개별 저장
    for noise_name in NOISE_LEVELS.keys():
        df_sub = df_noise[df_noise["noise"] == noise_name].copy()
        save_path = os.path.join(OUTPUT_DIR, f"features_{noise_name}.csv")
        df_sub.to_csv(save_path, index=False)
        print(f"[INFO] Saved: {save_path}")

    # orig도 따로 저장
    orig_save_path = os.path.join(OUTPUT_DIR, "features_orig.csv")
    df_orig.to_csv(orig_save_path, index=False)
    print(f"[INFO] Saved: {orig_save_path}")

    stats_tpv = summarize_by_noise_and_feature(df_all, "TPV22")
    stats_rmssd = summarize_by_noise_and_feature(df_all, "RMSSD")
    stats_all = pd.concat([stats_tpv, stats_rmssd], ignore_index=True)

    stats_csv = os.path.join(OUTPUT_DIR, "noise_robustness_stats.csv")
    stats_all.to_csv(stats_csv, index=False)
    print(f"[INFO] Saved stats CSV: {stats_csv}")

    save_boxplots(df_all)
    save_robustness_lineplots(stats_all)
    print(f"[INFO] Saved plots under: {os.path.join(OUTPUT_DIR, 'plots')}")

    print("\n===== ANOVA / Robustness Summary =====")
    print(stats_all.to_string(index=False))

[INFO] Loaded original CSV merged shape: (518, 13)
[INFO] Found 15 pkl files

[INFO] Noise level: alpha_0.01 (alpha=0.01)
[INFO] Processing S10 ...
[INFO] S10: extracted 36 windows
[INFO] Processing S11 ...
[INFO] S11: extracted 35 windows
[INFO] Processing S13 ...
[INFO] S13: extracted 35 windows
[INFO] Processing S14 ...
[INFO] S14: extracted 35 windows
[INFO] Processing S15 ...
[INFO] S15: extracted 35 windows
[INFO] Processing S16 ...
[INFO] S16: extracted 35 windows
[INFO] Processing S17 ...
[INFO] S17: extracted 36 windows
[INFO] Processing S2 ...
[INFO] S2: extracted 33 windows
[INFO] Processing S3 ...
[INFO] S3: extracted 34 windows
[INFO] Processing S4 ...
[INFO] S4: extracted 34 windows
[INFO] Processing S5 ...
[INFO] S5: extracted 33 windows
[INFO] Processing S6 ...
[INFO] S6: extracted 34 windows
[INFO] Processing S7 ...
[INFO] S7: extracted 35 windows
[INFO] Processing S8 ...
[INFO] S8: extracted 36 windows
[INFO] Processing S9 ...
[INFO] S9: extracted 32 windows

[INFO] N